# Day 22 — Query Optimisation Deep Dive

**What you will learn:**
- Catalyst optimizer: how Spark transforms your query plan
- Tungsten: binary encoding and code generation
- Predicate pushdown and projection pushdown
- Partition pruning — reading only relevant partitions
- Join strategies: when Spark chooses SortMergeJoin vs BroadcastHashJoin
- Key config knobs that affect query performance

**Exam relevance:** Optimisation (10%) + Architecture (20%) — the Catalyst pipeline, broadcast threshold, and partition pruning are tested.

In [ ]:
from pyspark.sql.functions import col, broadcast, count, avg, sum, round
import tempfile, os

LAKE = tempfile.mkdtemp(prefix="opt_lab_")
print(f"Lake: {LAKE}")

## 1. Catalyst Optimizer — The 4-Stage Pipeline

```
User query (DataFrame API / SQL)
    │
    ▼
1. Unresolved Logical Plan   — AST, names not yet resolved
    │
    ▼  (Analysis: resolve names against catalog)
2. Resolved Logical Plan     — all columns + types known
    │
    ▼  (Optimization: rule-based rewrites)
3. Optimized Logical Plan    — predicates pushed, constants folded
    │
    ▼  (Physical Planning: choose algorithms)
4. Physical Plan(s)          — SortMergeJoin vs BroadcastHashJoin, etc.
    │
    ▼  (Code Generation: Tungsten)
5. Generated JVM Bytecode    — executed
```

## 2. Predicate Pushdown

In [ ]:
data = [
    ("Alice", "Engineering", 95000, "RO"),
    ("Bob",   "Marketing",   72000, "RO"),
    ("Carol", "Engineering", 110000, "UK"),
    ("Dave",  "Sales",       58000,  "DE"),
    ("Eve",   "Marketing",   81000,  "RO"),
]
df = spark.createDataFrame(data, ["name", "dept", "salary", "country"])

# Write as Parquet so we can observe pushdown
path = os.path.join(LAKE, "employees")
df.write.mode("overwrite").parquet(path)

# Read back and apply filter
df_read = spark.read.parquet(path)

# Catalyst pushes the filter INTO the FileScan — avoids reading unnecessary rows
df_read.filter(col("salary") > 80000).explain("extended")

In [ ]:
# Projection pushdown — only read needed columns
# Parquet is columnar — unneeded columns are never read from disk
df_read.select("name", "salary").filter(col("salary") > 80000).explain("formatted")

## 3. Partition Pruning

In [ ]:
# Write partitioned by country
path_part = os.path.join(LAKE, "employees_partitioned")
df.write.mode("overwrite").partitionBy("country").parquet(path_part)

# Reading with a filter on the partition column — Spark skips irrelevant directories
pruned = spark.read.parquet(path_part).filter(col("country") == "RO")

# Physical plan shows PartitionFilters: [isnotnull(country), (country = RO)]
pruned.explain("formatted")

print("Rows (should be 3, only RO):", pruned.count())

## 4. Join Strategy Selection

Spark chooses join strategy based on data size estimates:

| Strategy | When chosen | How |
|---|---|---|
| BroadcastHashJoin | Small side ≤ `autoBroadcastJoinThreshold` (10MB default) | Small side sent to all executors |
| ShuffleHashJoin | Medium sizes, one side fits in memory | Hash table built on one side |
| SortMergeJoin | Both sides large | Both shuffled, sorted, then merged |
| BroadcastNestedLoopJoin | Cartesian or inequality join | Last resort — very slow |

In [ ]:
dept_df = spark.createDataFrame([
    ("Engineering", "Tech"),
    ("Marketing",   "Ops"),
    ("Sales",       "Ops"),
], ["dept", "division"])

print("Broadcast threshold:",
      spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Force SortMergeJoin by disabling broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("\n=== SortMergeJoin ===")
df.join(dept_df, on="dept").explain("simple")

# Restore and use explicit broadcast hint
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
print("\n=== BroadcastHashJoin ===")
df.join(broadcast(dept_df), on="dept").explain("simple")

## 5. Tungsten — Binary Processing

Tungsten operates below the SQL/DataFrame layer:
- **Unsafe Row format**: binary, off-heap, zero serialization for internal operations
- **Whole-stage code generation (WSCG)**: fuses multiple operators into a single JVM method
- **Vectorized reads**: reads Parquet in columnar batches, not row by row

You don't configure Tungsten directly — it's always on. But you benefit from it when:
- Using built-in functions (not UDFs)
- Reading columnar formats (Parquet, ORC)
- Operations that allow code generation (filter, project, aggregate)

In [ ]:
# Verify whole-stage codegen is enabled (default: true)
print("WSCG enabled:",
      spark.conf.get("spark.sql.codegen.wholeStage"))

print("Vectorized Parquet reader:",
      spark.conf.get("spark.sql.parquet.enableVectorizedReader"))

## 6. Optimisation Checklist for Real Jobs

1. **Push filters early** — filter before joins and aggregations
2. **Project early** — select only needed columns before expensive operations
3. **Broadcast small tables** — avoid shuffle on the small side
4. **Partition by query pattern** — partition columns used in WHERE = pruning
5. **Right shuffle partitions** — `spark.sql.shuffle.partitions` = 200 by default (often too high for small data, too low for huge data)
6. **Enable AQE** — Spark 3.x default, let it coalesce post-shuffle partitions
7. **Avoid UDFs** — Catalyst cannot optimise through them
8. **Cache shared results** — if used by multiple downstream actions

---

## Day 22 Checklist

- [ ] Know the 4 stages of the Catalyst optimizer pipeline
- [ ] Identified predicate pushdown in `explain()` output (filter inside FileScan)
- [ ] Verified partition pruning — `PartitionFilters` in physical plan
- [ ] Know when Spark chooses BroadcastHashJoin vs SortMergeJoin
- [ ] Disabled broadcast threshold to force SortMergeJoin, then used `broadcast()` hint
- [ ] Know that Tungsten's WSCG fuses operators — benefit by using built-ins, not UDFs

**Next:** Day 23 — AQE & Dynamic Partition Pruning